In [ ]:
import csv
import logging

logging.getLogger("requests").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

from collections import defaultdict
from itertools import chain
from rdflib.namespace import XSD
import re

from utils import wikiwrapper, splitname, Gender, checkRealIdentiesInKanto, unquoteWikipediaURL, qualifyName, NAMEPRIORITIES

import SPARQLWrapper
import time

In [ ]:
# check that labels contain only latin letters
# https://stackoverflow.com/questions/3094498/how-can-i-check-if-a-python-unicode-string-contains-non-western-letters

import unicodedata as ud

latin_letters= {}

def is_latin(uchr):
    try: return latin_letters[uchr]
    except KeyError:
         return latin_letters.setdefault(uchr, 'LATIN' in ud.name(uchr))

def only_roman_chars(unistr):
    return all(is_latin(uchr)
           for uchr in unistr
           if uchr.isalpha())

In [ ]:
DATATYPECONVERTERS = {
      str(XSD.integer):  int,
      # str(XSD.date):   lambda v: datetime.datetime.strptime(v, '%Y-%m-%d').date()
      str(XSD.decimal):  float,
    }

def convertDatatype(obj):
  return DATATYPECONVERTERS.get(obj.get('datatype'), str)(obj.get('value').strip())

def convertDatatypes(results):
    res = results["results"]["bindings"]
    return [dict([(k, convertDatatype(v)) for k,v in r.items()]) for r in res]


def requestDatabaseOffset(queryfnc : str, wrapper : SPARQLWrapper, res : list = [], limit : int = None, offset : int = None, resultlimit=2000):
    '''Performs a SPARQL query to a given endpoint. In a case of timeout, divides the list VALUES into smaller chuncks.
    queryfnc: a function returning the SPARQL query based by value list VALUES
    VALUES: see above
    resultlimit: limit the size of values block
    endpoint: default to Wikidata server
    arr: a list where the results are collected
    return: None, however results collected to parameter list 'arr'
    '''
    print(f'Quering limit {limit}, offset {offset}')
    try:
        if limit and limit>resultlimit:
            # if exceeded the 'resultlimit' value, chop the VALUES into smaller chunchs
            new_limit = limit//2
            requestDatabaseOffset(queryfnc, wrapper, res = res, limit = new_limit, offset = (offset or 0), resultlimit=resultlimit)
            requestDatabaseOffset(queryfnc, wrapper, res = res, limit = limit-new_limit, offset = (offset or 0) + new_limit, resultlimit=resultlimit)
        else:
            wrapper.setQuery(queryfnc(limit = limit, offset = offset))
            results = wrapper.query().convert()
            res += convertDatatypes(results)
            print(f"Currently {len(res)} results")
    except Exception as e:
        print(str(e))
        if '413 Client Error' in str(e):
            # if query failed, chop the VALUES into smaller chunchs
            new_limit = limit//2
            requestDatabaseOffset(queryfnc, wrapper, res = res, limit = new_limit, offset = (offset or 0), resultlimit=resultlimit)
            requestDatabaseOffset(queryfnc, wrapper, res = res, limit = limit-new_limit, offset = (offset or 0) + new_limit, resultlimit=resultlimit)

        elif '500 Server Error' in str(e) or \
            '403 Client Error' in str(e) or \
            'EndPointInternalError' in str(e):
            # sleep and retry
            if limit>1:
                print(f'Error "{str(e)}" - retry in 3 seconds')
                time.sleep(3)
                new_limit = limit//2
                requestDatabaseOffset(queryfnc, wrapper, res = res, limit = new_limit, offset = (offset or 0), resultlimit=resultlimit)
                requestDatabaseOffset(queryfnc, wrapper, res = res, limit = limit-new_limit, offset = (offset or 0) + new_limit, resultlimit=resultlimit)
        else:
            # here, ignore other error types
            print("Error {} occured.".format(e))
            print("Skipping query")
            pass # raise e


## Wikidata

In [ ]:
from utils import WIKIENDPOINT, requestDatabase

def wikidataPrefixes():
    return """PREFIX bd: <http://www.bigdata.com/rdf#> 
prefix p: <http://www.wikidata.org/prop/>
prefix pq: <http://www.wikidata.org/prop/qualifier/>
prefix ps: <http://www.wikidata.org/prop/statement/>
PREFIX mwapi: <https://www.mediawiki.org/ontology#API/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> 
PREFIX schema: <http://schema.org/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/> 
prefix wds: <http://www.wikidata.org/entity/statement/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> """

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_stdlib_context

### queries by different criteria

In [ ]:
query_results = []

In [ ]:
def query_wikidata_people(limit = None, offset = 0):
    '''Request database for actor details, for example see https://api.triplydb.com/s/CBNSqEDE7 '''
    return wikidataPrefixes() + """
      SELECT DISTINCT ?id ?idLabel ?gender 
        (YEAR(?birth_time) AS ?birth_year) 
        (YEAR(?death_time) AS ?death_year)
        ?ulan
      WITH {
        SELECT DISTINCT ?id 
        WHERE { ?id wdt:P31 wd:Q5 }
        """ + (f'LIMIT {limit} ' if limit else '') + (f'OFFSET {offset} ' if offset else '') + """
      } AS %i
      WHERE {
        INCLUDE %i
        OPTIONAL { ?id wdt:P569 ?birth_time }
        OPTIONAL { ?id wdt:P570 ?death_time }
        FILTER (BOUND(?birth_time) || BOUND(?death_time))

        OPTIONAL {
          VALUES (?gender ?gender_id) { ("female" wd:Q6581072) ("male" wd:Q6581097) } 
          ?id wdt:P21 ?gender_id
        }

        OPTIONAL { ?id wdt:P245 ?ulan }

        SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en" . }
      } """

for q in [query_wikidata_people]:
  requestDatabaseOffset(res = query_results, queryfnc = q,
                        limit=10_000_000,
                        wrapper = wikiwrapper)

print(f'{len(query_results)} results')
# 20000 results: 95.7s
# 1000000 results: 52m
# 4734975 results: 1983m
for ob in query_results[:10]:
    print(ob)

### stoplist for erroneous names

In [ ]:
stoplist = ["Alexis Sandelin",
            "Aleksis Sandelin"
            ]

In [ ]:
# filter by qualifying the given and family names
query_results = sorted(filter(
    lambda ob: qualifyName(ob.get('idLabel'), verbose = True, stoplist = stoplist) and \
        re.search(r'[A-ZÖÄÅČŠØÜŰ]', ob.get('idLabel', '')),
        query_results),
    key= lambda ob: ob.get('id'))

print(f'{len(query_results)} results')
# 141151 results

for ob in query_results[:20]:
  print(ob)

### precision of birth/death dates

In [ ]:
# Naive solution: year precision 1879-01-01 -> 1879
# NB & TODO: example about wikidata implementation for time precisions: https://api.triplydb.com/s/9_Y12BBcM
"""
for ob in query_results:
    for p in ['birth_year', 'death_year']:
        st = ob.get(p, '')
        if '-01-01' in st:
            ob[p] = st.replace('-01-01', '')
"""

### Unquote Wikipedia URLs

In [ ]:
fieldnames = ['id',
    'idLabel',
    'gender',
    'birth_year',
    'death_year',
    'ulan']

### check for entries marked with multiple genders

In [ ]:
check = defaultdict(set)
for ob in query_results:
    if ob.get('gender'):
        check[ob.get('Wikidata')].add(ob.get('gender'))

for k, arr in check.items():
    if len(arr)>1:
        print(k)
        print(f'\t{arr}')

### Export CSV

In [ ]:
CSVFILE = 'datasheets/wikidata.csv'
with open(CSVFILE, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile,
                            fieldnames = fieldnames,
                            delimiter = "\t")

    writer.writeheader()
    # for ob in wikires:
    for ob in sorted(query_results, key = lambda ob: ob.get('id')):
        writer.writerow(ob)

In [ ]:
'''
-rw-r--r--  1 ptleskin  staff  1455545 Jun  6 21:53 datasheets/wikidata.csv
   20235 datasheets/wikidata.csv
-rw-r--r--  1 ptleskin  staff  391609629 Jun  9 22:12 datasheets/wikidata.csv
 5348899 datasheets/wikidata.csv
'''


In [ ]:
! ls -alt datasheets/wikidata*.csv
! wc -l datasheets/wikidata*.csv
! head -n 10 datasheets/wikidata.csv

In [ ]:
! cut -f3 datasheets/wikidata.csv | sort | uniq